In [1]:
!pip install --upgrade numpy matplotlib pandas seaborn scipy --quiet


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
#!/usr/bin/env python3
"""
LLM PREDICTION ACCURACY VISUALIZATION
======================================
Analyzes Mean Absolute Error (MAE) between LLM predictions and perceived norms
across 24 climate policy questions.

Creates comprehensive visualizations showing:
1. MAE by question for each model
2. Overall model performance comparison
3. Prediction vs actual scatter plots
4. Error distribution analysis
"""

# ================================================================
# Fix matplotlib compatibility
# ================================================================
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")

print("="*80)
print("LLM PREDICTION ACCURACY ANALYSIS")
print("="*80)

# ================================================================
# 1. Load Data
# ================================================================

print("\n1. Loading data...")

df = pd.read_csv('pi_effects_llms_predictions.csv', encoding='latin-1')
print(f"   ✓ Loaded {len(df)} questions")

# ================================================================
# 2. Calculate MAE for Each Question
# ================================================================

print("\n2. Calculating Mean Absolute Error (MAE) with confidence intervals...")

models = ['gpt', 'claude', 'llama']  # Exclude Gemini (all missing)

# Calculate MAE for each model and question
mae_data = []

for idx, row in df.iterrows():
    qnum = row['qnum']
    perceived_norm = row['perceived norm']
    
    for model in models:
        pred_col = f'{model}_predicted'
        
        if pd.notna(row[pred_col]) and pd.notna(perceived_norm):
            prediction = row[pred_col]
            mae = abs(prediction - perceived_norm)
            
            mae_data.append({
                'qnum': qnum,
                'model': model.upper(),
                'prediction': prediction,
                'perceived_norm': perceived_norm,
                'mae': mae,
                'error': prediction - perceived_norm,  # Signed error
                'policy_type': row.get('type of PI', 'Unknown')
            })

mae_df = pd.DataFrame(mae_data)

print(f"   ✓ Calculated MAE for {len(mae_df)} predictions")

# Bootstrap confidence intervals for overall MAE
def bootstrap_mae_ci(errors, n_bootstrap=1000, confidence=0.95):
    """Calculate bootstrap confidence interval for MAE"""
    n = len(errors)
    bootstrap_maes = []
    
    for _ in range(n_bootstrap):
        sample = np.random.choice(errors, size=n, replace=True)
        bootstrap_maes.append(np.mean(np.abs(sample)))
    
    alpha = (1 - confidence) / 2
    lower = np.percentile(bootstrap_maes, alpha * 100)
    upper = np.percentile(bootstrap_maes, (1 - alpha) * 100)
    
    return lower, upper

# For single question CIs - use analytical approach since we only have 1 observation per question
def calculate_question_ci(mae_value, perceived_norm, n_questions=24):
    """
    Estimate confidence interval for individual question MAE
    Using approximate method based on overall variance
    """
    # Approximate CI based on typical variation
    # This is a rough estimate since we only have 1 prediction per question
    ci_width = mae_value * 0.3  # ~30% of MAE as rough CI
    lower = max(0, mae_value - ci_width)
    upper = mae_value + ci_width
    return lower, upper

# Calculate overall MAE with CIs
model_stats = []
for model in models:
    model_data = mae_df[mae_df['model'] == model.upper()]
    errors = model_data['error'].values
    mae_mean = model_data['mae'].mean()
    
    lower, upper = bootstrap_mae_ci(errors)
    
    model_stats.append({
        'model': model.upper(),
        'mae_mean': mae_mean,
        'mae_lower': lower,
        'mae_upper': upper
    })
    
    print(f"\n   {model.upper()}:")
    print(f"      MAE: {mae_mean:.2f} pp")
    print(f"      95% CI: [{lower:.2f}, {upper:.2f}]")

model_stats_df = pd.DataFrame(model_stats)

# Add CIs for each question (using standard error across models)
question_ci_data = []
for qnum in df['qnum'].unique():
    q_data = mae_df[mae_df['qnum'] == qnum]
    
    for model in ['GPT', 'CLAUDE', 'LLAMA']:
        model_q_data = q_data[q_data['model'] == model]
        
        if len(model_q_data) > 0:
            mae_val = model_q_data['mae'].iloc[0]
            perceived = model_q_data['perceived_norm'].iloc[0]
            
            # Use std across all questions for this model to estimate CI
            model_all = mae_df[mae_df['model'] == model]
            std_all = model_all['mae'].std()
            
            # Approximate CI using ±1 std (roughly 68% CI, conservative)
            lower = max(0, mae_val - std_all)
            upper = mae_val + std_all
            
            question_ci_data.append({
                'qnum': qnum,
                'model': model,
                'mae': mae_val,
                'mae_lower': lower,
                'mae_upper': upper
            })

question_ci_df = pd.DataFrame(question_ci_data)

# ================================================================
# 3. Create Visualizations
# ================================================================

print("\n3. Creating visualizations...")

# Create a large figure with multiple subplots using matplotlib 3.9+ syntax
fig = plt.figure()
fig.set_size_inches(20, 12)
fig.set_dpi(100)
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)  # Increased hspace from 0.3 to 0.35

# ----------------------------------------------------------------
# Plot 1: MAE by Question for Each Model
# ----------------------------------------------------------------

ax1 = fig.add_subplot(gs[0, :])

# Pivot data for grouped bar chart
mae_pivot = mae_df.pivot(index='qnum', columns='model', values='mae')

x = np.arange(len(mae_pivot))
width = 0.25

for i, model in enumerate(['GPT', 'CLAUDE', 'LLAMA']):
    if model in mae_pivot.columns:
        offset = (i - 1) * width
        ax1.bar(x + offset, mae_pivot[model], width, label=model, alpha=0.8)

ax1.set_xlabel('Question Number', fontsize=11, fontweight='bold')
ax1.set_ylabel('MAE (percentage points)', fontsize=11, fontweight='bold')
ax1.set_title('Mean Absolute Error by Question', fontsize=13, fontweight='bold', pad=15)
ax1.set_xticks(x)
ax1.set_xticklabels([f'Q{i}' for i in mae_pivot.index], fontsize=8)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# ----------------------------------------------------------------
# Plot 2: Overall Model Performance Comparison with CIs
# ----------------------------------------------------------------

ax2 = fig.add_subplot(gs[1, 0])

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# Calculate error bars (CI width)
y_err_lower = model_stats_df['mae_mean'] - model_stats_df['mae_lower']
y_err_upper = model_stats_df['mae_upper'] - model_stats_df['mae_mean']

ax2.bar(model_stats_df['model'], model_stats_df['mae_mean'], 
        yerr=[y_err_lower, y_err_upper], capsize=10, alpha=0.7, color=colors,
        error_kw={'linewidth': 2, 'elinewidth': 2})

ax2.set_ylabel('MAE (pp)', fontsize=10, fontweight='bold')  # Shortened label
ax2.set_title('Overall Model Performance\n(Mean with 95% Bootstrap CI)', fontsize=12, fontweight='bold', pad=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.tick_params(axis='y', labelsize=8)
ax2.tick_params(axis='x', labelsize=9)

# Add value labels on bars - positioned higher to avoid overlap
for i, (idx, row) in enumerate(model_stats_df.iterrows()):
    ax2.text(i, row['mae_upper'] + 1.5, 
            f"{row['mae_mean']:.1f}\n[{row['mae_lower']:.1f}, {row['mae_upper']:.1f}]", 
            ha='center', fontsize=7, fontweight='bold')

# ----------------------------------------------------------------
# Plot 3: GPT - Prediction vs Actual with CI
# ----------------------------------------------------------------

ax3 = fig.add_subplot(gs[1, 1])

gpt_data = mae_df[mae_df['model'] == 'GPT']
ax3.scatter(gpt_data['perceived_norm'], gpt_data['prediction'], 
           alpha=0.6, s=100, color='#1f77b4')
ax3.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect prediction')

# Add shaded region for ±10pp error
ax3.fill_between([0, 100], [0, 100], [10, 110], alpha=0.1, color='green', label='±10pp')
ax3.fill_between([0, 100], [-10, 90], [0, 100], alpha=0.1, color='green')

ax3.set_xlabel('Perceived Norm (%)', fontsize=10, fontweight='bold')
ax3.set_ylabel('GPT Prediction (%)', fontsize=10, fontweight='bold')
ax3.set_title('GPT: Predicted vs Actual', fontsize=12, fontweight='bold', pad=10)
ax3.legend(fontsize=8, loc='upper left')
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, 100)
ax3.set_ylim(0, 100)
ax3.tick_params(axis='both', labelsize=8)

# Add correlation and MAE
if len(gpt_data) > 0:
    r, p = pearsonr(gpt_data['perceived_norm'], gpt_data['prediction'])
    mae = gpt_data['mae'].mean()
    gpt_ci = model_stats_df[model_stats_df['model'] == 'GPT'].iloc[0]
    ax3.text(0.05, 0.95, 
            f'r = {r:.3f}\np = {p:.4f}\nMAE = {mae:.1f}pp\n95% CI: [{gpt_ci["mae_lower"]:.1f}, {gpt_ci["mae_upper"]:.1f}]', 
            transform=ax3.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), fontsize=8)

# ----------------------------------------------------------------
# Plot 4: Claude - Prediction vs Actual with CI
# ----------------------------------------------------------------

ax4 = fig.add_subplot(gs[1, 2])

claude_data = mae_df[mae_df['model'] == 'CLAUDE']
ax4.scatter(claude_data['perceived_norm'], claude_data['prediction'], 
           alpha=0.6, s=100, color='#ff7f0e')
ax4.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect prediction')

# Add shaded region for ±10pp error
ax4.fill_between([0, 100], [0, 100], [10, 110], alpha=0.1, color='green', label='±10pp')
ax4.fill_between([0, 100], [-10, 90], [0, 100], alpha=0.1, color='green')

ax4.set_xlabel('Perceived Norm (%)', fontsize=10, fontweight='bold')
ax4.set_ylabel('Claude Prediction (%)', fontsize=10, fontweight='bold')
ax4.set_title('Claude: Predicted vs Actual', fontsize=12, fontweight='bold', pad=10)
ax4.legend(fontsize=8, loc='upper left')
ax4.grid(True, alpha=0.3)
ax4.set_xlim(0, 100)
ax4.set_ylim(0, 100)
ax4.tick_params(axis='both', labelsize=8)

# Add correlation and MAE
if len(claude_data) > 0:
    r, p = pearsonr(claude_data['perceived_norm'], claude_data['prediction'])
    mae = claude_data['mae'].mean()
    claude_ci = model_stats_df[model_stats_df['model'] == 'CLAUDE'].iloc[0]
    ax4.text(0.05, 0.95, 
            f'r = {r:.3f}\np = {p:.4f}\nMAE = {mae:.1f}pp\n95% CI: [{claude_ci["mae_lower"]:.1f}, {claude_ci["mae_upper"]:.1f}]', 
            transform=ax4.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), fontsize=8)

# ----------------------------------------------------------------
# Plot 5: Llama - Prediction vs Actual with CI
# ----------------------------------------------------------------

ax5 = fig.add_subplot(gs[2, 0])

llama_data = mae_df[mae_df['model'] == 'LLAMA']
ax5.scatter(llama_data['perceived_norm'], llama_data['prediction'], 
           alpha=0.6, s=100, color='#2ca02c')
ax5.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect prediction')

# Add shaded region for ±10pp error
ax5.fill_between([0, 100], [0, 100], [10, 110], alpha=0.1, color='green', label='±10pp')
ax5.fill_between([0, 100], [-10, 90], [0, 100], alpha=0.1, color='green')

ax5.set_xlabel('Perceived Norm (%)', fontsize=10, fontweight='bold')
ax5.set_ylabel('Llama Prediction (%)', fontsize=10, fontweight='bold')
ax5.set_title('Llama: Predicted vs Actual', fontsize=12, fontweight='bold', pad=10)
ax5.legend(fontsize=8, loc='upper left')
ax5.grid(True, alpha=0.3)
ax5.set_xlim(0, 100)
ax5.set_ylim(0, 100)
ax5.tick_params(axis='both', labelsize=8)

# Add correlation and MAE
if len(llama_data) > 0:
    r, p = pearsonr(llama_data['perceived_norm'], llama_data['prediction'])
    mae = llama_data['mae'].mean()
    llama_ci = model_stats_df[model_stats_df['model'] == 'LLAMA'].iloc[0]
    ax5.text(0.05, 0.95, 
            f'r = {r:.3f}\np = {p:.4f}\nMAE = {mae:.1f}pp\n95% CI: [{llama_ci["mae_lower"]:.1f}, {llama_ci["mae_upper"]:.1f}]', 
            transform=ax5.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), fontsize=8)

# ----------------------------------------------------------------
# Plot 6: Error Distribution (Signed Errors)
# ----------------------------------------------------------------

ax6 = fig.add_subplot(gs[2, 1])

for model, color in zip(['GPT', 'CLAUDE', 'LLAMA'], colors):
    model_errors = mae_df[mae_df['model'] == model]['error']
    ax6.hist(model_errors, bins=15, alpha=0.5, label=model, color=color)

ax6.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No bias')
ax6.set_xlabel('Prediction Error (pp)', fontsize=10, fontweight='bold')
ax6.set_ylabel('Frequency', fontsize=10, fontweight='bold')
ax6.set_title('Error Distribution\n(Positive = Overprediction)', fontsize=12, fontweight='bold', pad=10)
ax6.legend(fontsize=8)
ax6.grid(True, alpha=0.3, axis='y')
ax6.tick_params(axis='both', labelsize=8)

# ----------------------------------------------------------------
# Plot 7: Bias Analysis (Mean Error by Model)
# ----------------------------------------------------------------

ax7 = fig.add_subplot(gs[2, 2])

bias_data = mae_df.groupby('model')['error'].agg(['mean', 'std']).reset_index()

ax7.barh(bias_data['model'], bias_data['mean'], 
        xerr=bias_data['std'], capsize=10, alpha=0.7, color=colors)
ax7.axvline(x=0, color='red', linestyle='--', linewidth=2)

ax7.set_xlabel('Mean Error (pp)', fontsize=10, fontweight='bold')
ax7.set_title('Model Bias\n(Positive = Tends to Overpredict)', fontsize=12, fontweight='bold', pad=10)
ax7.grid(True, alpha=0.3, axis='x')
ax7.tick_params(axis='both', labelsize=8)

# Add value labels - positioned to avoid overlap
for i, (idx, row) in enumerate(bias_data.iterrows()):
    x_pos = row['mean'] + (row['std'] * 1.2 if row['mean'] > 0 else -row['std'] * 1.2)
    ax7.text(x_pos, i, f"  {row['mean']:.1f}", 
            va='center', fontsize=8, fontweight='bold')

plt.suptitle('LLM Climate Policy Prediction Accuracy Analysis', 
            fontsize=15, fontweight='bold', y=0.995)

plt.savefig('llm_prediction_accuracy.pdf', dpi=300, bbox_inches='tight')
print("   ✓ Saved llm_prediction_accuracy.pdf")
plt.savefig('llm_prediction_accuracy.png', dpi=300, bbox_inches='tight')
print("   ✓ Saved llm_prediction_accuracy.png")
plt.close()

# ================================================================
# 4. Create Detailed MAE Table
# ================================================================

print("\n4. Creating detailed results table...")

# Create summary table
summary_table = mae_df.pivot_table(
    index='qnum',
    columns='model',
    values='mae',
    aggfunc='mean'
).round(2)

summary_table['Min_MAE'] = summary_table.min(axis=1)
summary_table['Max_MAE'] = summary_table.max(axis=1)
summary_table['Range'] = summary_table['Max_MAE'] - summary_table['Min_MAE']

# Add perceived norm
summary_table = summary_table.merge(
    df[['qnum', 'perceived norm', 'type of PI']].set_index('qnum'),
    left_index=True,
    right_index=True,
    how='left'
)

# Reorder columns
cols = ['perceived norm', 'GPT', 'CLAUDE', 'LLAMA', 'Min_MAE', 'Max_MAE', 'Range', 'type of PI']
summary_table = summary_table[[col for col in cols if col in summary_table.columns]]

summary_table.to_csv('llm_mae_by_question.csv')
print("   ✓ Saved llm_mae_by_question.csv")

# ================================================================
# 5. Print Summary Statistics
# ================================================================

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print("\n📊 Overall Performance (with 95% Bootstrap Confidence Intervals):")
for _, row in model_stats_df.iterrows():
    model = row['model']
    mae_mean = row['mae_mean']
    mae_lower = row['mae_lower']
    mae_upper = row['mae_upper']
    
    model_data = mae_df[mae_df['model'] == model]
    bias = model_data['error'].mean()
    
    print(f"\n{model}:")
    print(f"   MAE: {mae_mean:.2f} pp")
    print(f"   95% CI: [{mae_lower:.2f}, {mae_upper:.2f}]")
    print(f"   CI Width: {mae_upper - mae_lower:.2f} pp")
    print(f"   Bias: {bias:+.2f} pp ({'overpredicts' if bias > 0 else 'underpredicts'})")
    print(f"   Correlation: r = {pearsonr(model_data['perceived_norm'], model_data['prediction'])[0]:.3f}")

print("\n🎯 Best Predictions (Lowest MAE):")
best_predictions = mae_df.nsmallest(5, 'mae')[['qnum', 'model', 'prediction', 'perceived_norm', 'mae']]
print(best_predictions.to_string(index=False))

print("\n⚠️  Worst Predictions (Highest MAE):")
worst_predictions = mae_df.nlargest(5, 'mae')[['qnum', 'model', 'prediction', 'perceived_norm', 'mae']]
print(worst_predictions.to_string(index=False))

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nFiles created:")
print("   1. llm_prediction_accuracy.pdf - Comprehensive visualization (PDF)")
print("   2. llm_prediction_accuracy.png - Comprehensive visualization (PNG)")
print("   3. llm_mae_by_question.csv - Detailed MAE table")
print("\n" + "="*80)

LLM PREDICTION ACCURACY ANALYSIS

1. Loading data...
   ✓ Loaded 24 questions

2. Calculating Mean Absolute Error (MAE) with confidence intervals...
   ✓ Calculated MAE for 72 predictions

   GPT:
      MAE: 17.02 pp
      95% CI: [14.25, 19.88]

   CLAUDE:
      MAE: 8.30 pp
      95% CI: [6.25, 10.53]

   LLAMA:
      MAE: 3.71 pp
      95% CI: [2.73, 4.78]

3. Creating visualizations...
   ✓ Saved llm_prediction_accuracy.pdf
   ✓ Saved llm_prediction_accuracy.png

4. Creating detailed results table...
   ✓ Saved llm_mae_by_question.csv

SUMMARY STATISTICS

📊 Overall Performance (with 95% Bootstrap Confidence Intervals):

GPT:
   MAE: 17.02 pp
   95% CI: [14.25, 19.88]
   CI Width: 5.64 pp
   Bias: +17.02 pp (overpredicts)
   Correlation: r = -0.191

CLAUDE:
   MAE: 8.30 pp
   95% CI: [6.25, 10.53]
   CI Width: 4.28 pp
   Bias: +5.70 pp (overpredicts)
   Correlation: r = 0.282

LLAMA:
   MAE: 3.71 pp
   95% CI: [2.73, 4.78]
   CI Width: 2.05 pp
   Bias: +0.36 pp (overpredicts)
   Cor

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>